# Assignment - Sentiment Analysis on Tweet

I have cleared the output before uploading the file into the Github becasue I was encountering a technical error while uploading this file with output into the Github repository.

So please consider this and don't deduct marks for not adding the output in the file.

I will add the screenshots of the ouput when I upload the link in Paatshala as proof.

Step 1 : Import Required Libraries

In [ ]:
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Baseline model
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import pickle

# BERT
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments

Step 2 : Load Dataset

In [ ]:
# Load the dataset tweet.csv
df = pd.read_csv("/content/tweets.csv")

In [ ]:
# Use only required columns by removing the unwanted 'id' column
df = df[['label', 'tweet']]

In [ ]:
# Display the first few records
df.head()

In [ ]:
# Check dataset info
df.info()

Step 3 : Data Preprocessing

In [ ]:
# Defining a function to clean tweets which contains URLS, hashtags, @ symbols, punctuations etc.
def clean_tweets(tweet):
    tweet = tweet.lower()  # Convert to lowercase

    tweet = re.sub(r"http\S+", "", tweet)  # This is to remove URLs from the tweets
    tweet = re.sub(r"@\w+", "", tweet)     # This is to remove the mentions like @names or ids
    tweet = re.sub(r"#", "", tweet)        # This is to remove the hashtag symbols

    tweet = re.sub(r"[^\w\s]", "", tweet)  # This is for removing the punctuations
    return tweet

In [ ]:
# Apply cleaning
df['tweet'] = df['tweet'].apply(clean_tweets)

In [ ]:
# Display first few records of cleaned tweet column
df[['label','tweet']].head()

Step 4 : Train-Test Split

In [ ]:
# Split the dataset to X and y
X = df['tweet']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# MODEL 1: TF-IDF + Logistic Regression

Step 5 : Feature Extraction

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english')

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

Step 6 : Train Model

In [ ]:
model_lr = LogisticRegression()
model_lr.fit(X_train_tfidf, y_train)

Step 7 : Evaluate Model

In [ ]:
y_pred_lr = model_lr.predict(X_test_tfidf)

accuracy_lr = accuracy_score(y_test, y_pred_lr)
print("Logistic Regression Accuracy:", accuracy_lr)

Step 8 : Save the Model

In [ ]:
pickle.dump(model_lr, open("logistic_model.pkl", "wb"))
pickle.dump(vectorizer, open("tfidf_vectorizer.pkl", "wb"))

# MODEL 2: BERT (Advanced Model)

Step 9 : Prepare Data

In [ ]:
train_texts = X_train.tolist()
test_texts = X_test.tolist()

train_labels = y_train.tolist()
test_labels = y_test.tolist()

Step 10 : Tokenization

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_encodings = tokenizer(train_texts, truncation=True, padding=True)
test_encodings = tokenizer(test_texts, truncation=True, padding=True)

Step 11 : Dataset Class

In [ ]:
class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TweetDataset(train_encodings, train_labels)
test_dataset = TweetDataset(test_encodings, test_labels)

Step 12 : Load BERT Model

In [ ]:
model_bert = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=2
)

Step 13 : Training Setup

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    logging_steps=10
)

Step 14 : Train Model

In [ ]:
trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

Step 15 : Evaluate BERT

In [ ]:
predictions = trainer.predict(test_dataset)

y_pred_bert = predictions.predictions.argmax(axis=1)

accuracy_bert = accuracy_score(test_labels, y_pred_bert)
print("BERT Accuracy:", accuracy_bert)

Step 16 : Save BERT Model

In [ ]:
model_bert.save_pretrained("bert_model")
tokenizer.save_pretrained("bert_model")

FINAL COMPARISON

In [ ]:
print("\nFinal Comparison:")
print("Logistic Regression Accuracy:", accuracy_lr)
print("BERT Accuracy:", accuracy_bert)

Step 17 : Load Saved BERT Model

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

# Load saved model
model = BertForSequenceClassification.from_pretrained("bert_model")
tokenizer = BertTokenizer.from_pretrained("bert_model")

model.eval()

Step 18 : Create Prediction Function

In [ ]:
def predict_sentiment(tweet):
    tweet = clean_tweets(tweet)

    inputs = tokenizer(tweet, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).item()

    return "Positive" if pred == 0 else "Negative"

Step 19 : Test with Sample Data

In [ ]:
test_tweets = [
    "I love this new phone, battery life is good",
    "This laptop is very slow and performance is bad"
]

for tweet in test_tweets:
    print("Tweet:", tweet)
    print("Predicted Sentiment:", predict_sentiment(tweet))